## Mehreen Ali Gillani
### Project 1
### Recommender System

### Step 1: Choose a dataset and define the business context

Business context: "This system recommends movies to users based on their rating patterns."

I've created a small, manageable dataset with 6 users and 6 movies, including some missing values so I can verify calculations by hand.

### Step 2: Create the dataset

Let's create a user-item matrix with ratings from 1-5:

Users: Alice, Bob, Carol, David, Emma, Frank
Movies: Inception, Titanic, Avatar, The Matrix, Gladiator, Frozen

Here's the rating matrix (0 or blank = missing):

In [18]:
# DATA 612 Project 1 - Movie Recommender System
# Global Baseline Predictors and RMSE

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Step 1: Create the user-item matrix
print("="*60)
print("STEP 1: Creating User-Item Matrix")
print("="*60)

# Create the data
data = {
    'User': ['Alice', 'Bob', 'Carol', 'David', 'Emma', 'Frank'],
    'Inception': [5, 4, 0, 3, 5, 0],
    'Titanic': [4, 0, 5, 3, 4, 2],
    'Avatar': [0, 5, 4, 0, 5, 3],
    'The Matrix': [5, 4, 0, 4, 0, 3],
    'Gladiator': [3, 0, 5, 4, 0, 5],
    'Frozen': [0, 2, 3, 0, 4, 0]
}

# Convert to DataFrame
df = pd.DataFrame(data)
df.set_index('User', inplace=True)

print("\nUser-Item Matrix (0 indicates missing rating):")
print(df)
print(f"\nShape: {df.shape} (users x movies)")

STEP 1: Creating User-Item Matrix

User-Item Matrix (0 indicates missing rating):
       Inception  Titanic  Avatar  The Matrix  Gladiator  Frozen
User                                                            
Alice          5        4       0           5          3       0
Bob            4        0       5           4          0       2
Carol          0        5       4           0          5       3
David          3        3       0           4          4       0
Emma           5        4       5           0          0       4
Frank          0        2       3           3          5       0

Shape: (6, 6) (users x movies)


In [26]:
# Step 2: Convert to long format and split into train/test
print("\n" + "="*60)
print("STEP 2: Converting to Long Format and Creating Train/Test Split")
print("="*60)

# Convert from wide to long format
long_df = df.reset_index().melt(
    id_vars=['User'], 
    var_name='Movie', 
    value_name='Rating'
)

# Remove missing ratings (where Rating = 0)
long_df = long_df[long_df['Rating'] > 0].copy()



# Split into training (80%) and testing (20%)
np.random.seed(42)  # For reproducibility
train_df, test_df = train_test_split(
    long_df, 
    test_size=0.2, 
    random_state=42
)
print(f"\nTotal ratings available: {len(long_df)}")
print("\nRows of training set:")
print(train_df.head(19))

print(f"\nTraining set size: {len(train_df)} ratings")
print(f"Test set size: {len(test_df)} ratings")

print("\nRows of test set:")
print(test_df.head(5))


STEP 2: Converting to Long Format and Creating Train/Test Split

Total ratings available: 24

Rows of training set:
     User       Movie  Rating
13    Bob      Avatar       5
18  Alice  The Matrix       5
1     Bob   Inception       4
31    Bob      Frozen       2
8   Carol     Titanic       5
3   David   Inception       3
17  Frank      Avatar       3
21  David  The Matrix       4
4    Emma   Inception       5
6   Alice     Titanic       4
32  Carol      Frozen       3
24  Alice   Gladiator       3
29  Frank   Gladiator       5
34   Emma      Frozen       4
10   Emma     Titanic       4
14  Carol      Avatar       4
19    Bob  The Matrix       4
27  David   Gladiator       4
9   David     Titanic       3

Training set size: 19 ratings
Test set size: 5 ratings

Rows of test set:
     User       Movie  Rating
11  Frank     Titanic       2
23  Frank  The Matrix       3
0   Alice   Inception       5
26  Carol   Gladiator       5
16   Emma      Avatar       5


In [20]:
# Step 3: Calculate raw average (global mean)
print("\n" + "="*60)
print("STEP 3: Raw Average (Global Mean)")
print("="*60)

global_mean = train_df['Rating'].mean()
print(f"\nGlobal mean rating from training data: {global_mean:.3f}")

# Raw average prediction is just the global mean for all user-item pairs
print("\nRaw average prediction: Same value ({:.3f}) for ALL user-item combinations".format(global_mean))


STEP 3: Raw Average (Global Mean)

Global mean rating from training data: 3.895

Raw average prediction: Same value (3.895) for ALL user-item combinations


In [13]:
# Step 4: Calculate RMSE for raw average
print("\n" + "="*60)
print("STEP 4: Calculating RMSE for Raw Average")
print("="*60)

def calculate_rmse(actual_ratings, predicted_ratings):
    """Calculate Root Mean Square Error"""
    return np.sqrt(np.mean((actual_ratings - predicted_ratings) ** 2))

# Calculate RMSE for training data
train_pred_raw = [global_mean] * len(train_df)
train_rmse_raw = calculate_rmse(train_df['Rating'].values, train_pred_raw)

# Calculate RMSE for test data
test_pred_raw = [global_mean] * len(test_df)
test_rmse_raw = calculate_rmse(test_df['Rating'].values, test_pred_raw)

print(f"\nRaw Average RMSE - Training: {train_rmse_raw:.4f}")
print(f"Raw Average RMSE - Test: {test_rmse_raw:.4f}")




STEP 4: Calculating RMSE for Raw Average

Raw Average RMSE - Training: 0.8519
Raw Average RMSE - Test: 1.2693


**Training RMSE: 0.8519** - On average, the global mean prediction is off by about 0.85 rating points for ratings the model has already seen.

**Test RMSE: 1.2693** - On average, the global mean prediction is off by about 1.27 rating points for unseen ratings.

**Key insight:** The test RMSE is 49% higher than training RMSE. This makes sense because:

-   The global mean is calculated from training data

-   Test data contains different user-item combinations with their own patterns

In [9]:
# Step 5: Calculate user and item biases
print("\n" + "="*60)
print("STEP 5: Calculating User and Item Biases")
print("="*60)

# Calculate global mean from training data
global_mean = train_df['Rating'].mean()

# Calculate user biases
user_avg = train_df.groupby('User')['Rating'].mean()
user_biases = user_avg - global_mean

# Calculate item biases
item_avg = train_df.groupby('Movie')['Rating'].mean()
item_biases = item_avg - global_mean

print(f"\nUser Biases (deviation from global mean):")
for user, bias in user_biases.items():
    print(f"  {user}: {bias:+.3f}")

print(f"\nItem Biases (deviation from global mean):")
for movie, bias in item_biases.items():
    print(f"  {movie}: {bias:+.3f}")

print("\nNote: A positive bias means user/item tends to rate higher than average,")
print("negative bias means they tend to rate lower than average.")


STEP 5: Calculating User and Item Biases

User Biases (deviation from global mean):
  Alice: +0.105
  Bob: -0.145
  Carol: +0.105
  David: -0.395
  Emma: +0.439
  Frank: +0.105

Item Biases (deviation from global mean):
  Avatar: +0.105
  Frozen: -0.895
  Gladiator: +0.105
  Inception: +0.105
  The Matrix: +0.439
  Titanic: +0.105

Note: A positive bias means user/item tends to rate higher than average,
negative bias means they tend to rate lower than average.


#### Summary of User and Item Biases

**User Biases (Who rates high vs. low):**

-   Emma (+0.44) is the most generous rater — consistently rates ~½ star above average

-   David (-0.40) is the toughest critic — rates nearly ½ star below average

-   Alice, Carol, Frank (+0.11) are slightly above-average raters
-   Bob (-0.15) is mildly below average

**Item Biases (Which movies are liked vs. disliked):**

-   The Matrix (+0.44) is the most popular movie among users

-   Frozen (-0.90) is strongly disliked — rates nearly a full star below average

-   All other movies (+0.11) are slightly above average in popularity

**Key Insight:**

The wide range of biases (from -0.90 to +0.44) indicates that both users and items vary significantly in their rating tendencies. This explains why the raw average performed poorly, it couldn't account for David being a harsh critic or Frozen being universally disliked.

In [14]:
# Step 6: Calculate baseline predictors
print("\n" + "="*60)
print("STEP 6: Calculating Baseline Predictors")
print("="*60)

def calculate_baseline_predictor(user, movie, global_mean, user_biases, item_biases):
    """
    Baseline prediction = global mean + user bias + item bias
    """
    user_bias = user_biases.get(user, 0)
    item_bias = item_biases.get(movie, 0)
    
    prediction = global_mean + user_bias + item_bias
    
    # Clip to valid rating range (1-5)
    return np.clip(prediction, 1, 5)

# Calculate baseline predictions for training set
train_baseline_pred = []
for idx, row in train_df.iterrows():
    pred = calculate_baseline_predictor(
        row['User'], row['Movie'], 
        global_mean, user_biases, item_biases
    )
    train_baseline_pred.append(pred)

# Calculate baseline predictions for test set
test_baseline_pred = []
for idx, row in test_df.iterrows():
    pred = calculate_baseline_predictor(
        row['User'], row['Movie'], 
        global_mean, user_biases, item_biases
    )
    test_baseline_pred.append(pred)

print("\nExample predictions for first 5 test set items:")
for i in range(min(5, len(test_df))):
    row = test_df.iloc[i]
    print(f"  {row['User']} - {row['Movie']}: "
          f"Actual={row['Rating']}, "
          f"Predicted={test_baseline_pred[i]:.2f}")


STEP 6: Calculating Baseline Predictors

Example predictions for first 5 test set items:
  Frank - Titanic: Actual=2, Predicted=4.11
  Frank - The Matrix: Actual=3, Predicted=4.44
  Alice - Inception: Actual=5, Predicted=4.11
  Carol - Gladiator: Actual=5, Predicted=4.11
  Emma - Avatar: Actual=5, Predicted=4.44


**Observation:** All predicted ratings are clustered between 4.11 and 4.44, while actual ratings range from 2 to 5.

**Key Issues Identified:**

**Frank's predictions are too high:**

-   Predicted Titanic at 4.11, but Frank actually rated it 2 (off by 2.11 stars)

-   Predicted The Matrix at 4.44, actual was 3 (off by 1.44 stars)

-   Frank's bias (+0.11) doesn't capture his tendency to rate some movies very low

**Model is overly optimistic:**

-   No predictions below 4.11, but actual includes ratings of 2 and 3

-   The Frozen bias (-0.90) isn't appearing in these test samples

**Lack of variation:**

-   Different user-movie combinations get nearly identical predictions

-   Actual ratings show much wider spread (2 to 5)

**Root Cause:** The model only adds user and item biases to the global mean, but interactions between specific users and items matter. 
Frank might dislike Titanic specifically, not all movies, this requires more sophisticated models. 

In [15]:
# Step 7: Calculate RMSE for baseline predictors
print("\n" + "="*60)
print("STEP 7: Calculating RMSE for Baseline Predictors")
print("="*60)

train_rmse_baseline = calculate_rmse(train_df['Rating'].values, train_baseline_pred)
test_rmse_baseline = calculate_rmse(test_df['Rating'].values, test_baseline_pred)

print(f"\nBaseline Predictors RMSE - Training: {train_rmse_baseline:.4f}")
print(f"Baseline Predictors RMSE - Test: {test_rmse_baseline:.4f}")


STEP 7: Calculating RMSE for Baseline Predictors

Baseline Predictors RMSE - Training: 0.6572
Baseline Predictors RMSE - Test: 1.2975


**Key Finding:** Baseline predictors improved training fit but failed to generalize to test data (slightly worse than raw average).


**Why?** The model is **overfitting** — user/item biases learned from training data don't match the patterns in test data. 
The small dataset (only ~24 training ratings) makes biases unstable. Frank's +0.11 bias wrongly 

inflated his test predictions (actual ratings were 2-3).

**Conclusion:** While biases help in theory, this dataset is too small to benefit. Need more data.

In [17]:
# Step 8: Summary and comparison
print("\n" + "="*60)
print("STEP 8: Results Summary")
print("="*60)

print("\nRMSE Comparison:")
print("-" * 40)
print(f"{'Method':<25} {'Training':<12} {'Test':<12}")
print("-" * 40)
print(f"{'Raw Average (Global Mean)':<25} {train_rmse_raw:<12.4f} {test_rmse_raw:<12.4f}")
print(f"{'Baseline Predictors':<25} {train_rmse_baseline:<12.4f} {test_rmse_baseline:<12.4f}")
print("-" * 40)

print("\nIMPROVEMENT:")
improvement = ((test_rmse_raw - test_rmse_baseline) / test_rmse_raw) * 100
print(f"Baseline predictors reduced test RMSE by {improvement:.1f}%")



STEP 8: Results Summary

RMSE Comparison:
----------------------------------------
Method                    Training     Test        
----------------------------------------
Raw Average (Global Mean) 0.8519       1.2693      
Baseline Predictors       0.6572       1.2975      
----------------------------------------

IMPROVEMENT:
Baseline predictors reduced test RMSE by -2.2%


Baseline predictors failed — test RMSE got worse (-2.2%)

**Why it failed:**

-   Overfitting on small dataset (only ~24 training ratings)

-   User/item biases unstable with sparse data

-   Frank's +0.11 bias incorrectly predicted his low ratings (2-3) as ~4.2

**Key takeaway:** Simple bias correction works in theory but requires sufficient data. With tiny datasets, raw average can actually generalize better.
